In [3]:
from pathlib import Path
import zipfile
import re
import pandas as pd
import numpy as np

# ---------- KONFIGURASI ----------
ZIP_PATH = Path("SCRIPT.zip")  # ubah ke path file ZIP kamu
OUTPUT_DIR = Path("latic_ground_truth_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- MEMBACA FILE SCRIPT ----------
def read_script_file(zf, member):
    """Baca file TXT berformat: Audio_ID<TAB>teks/notasi."""
    text = zf.read(member).decode("utf-8-sig", errors="replace")
    records = {}
    for line in text.splitlines():
        if not line.strip():
            continue
        parts = line.split("\t", 1)
        if len(parts) != 2:
            continue
        audio_id = parts[0].strip()
        value = parts[1].strip()
        if audio_id:
            records[audio_id] = value
    return records

def get_member(zf, folder, speaker_file):
    """Cari path member dengan toleransi nama folder yang sedikit berbeda."""
    candidates = [
        n for n in zf.namelist()
        if n.lower().endswith(f"/{folder.lower()}/{speaker_file.lower()}")
    ]
    if not candidates:
        # Toleransi typo nama folder Orignial_Notations pada ZIP sumber.
        folder_norm = folder.lower().replace("original", "orignial")
        candidates = [
            n for n in zf.namelist()
            if n.lower().endswith(f"/{folder_norm}/{speaker_file.lower()}")
        ]
    if not candidates:
        raise FileNotFoundError(f"Tidak menemukan {folder}/{speaker_file} di dalam ZIP.")
    return candidates[0]

def load_all_scripts(zip_path):
    if not zip_path.exists():
        raise FileNotFoundError(
            f"ZIP tidak ditemukan: {zip_path.resolve()}\n"
            "Letakkan SCRIPT.zip di folder kerja atau ubah ZIP_PATH."
        )
    all_records = {}
    with zipfile.ZipFile(zip_path, "r") as zf:
        speaker_files = sorted({
            Path(n).name for n in zf.namelist()
            if n.lower().endswith(".txt")
        })
        for speaker_file in speaker_files:
            folders = {
                "original_hanzi": "Original_Text_Script",
                "original_pinyin": "Orignial_Notations",
                "testing_pinyin": "Testing_Notations",
                "testing_hanzi": "Testing_Text_Script",
            }
            maps = {}
            for key, folder in folders.items():
                member = get_member(zf, folder, speaker_file)
                maps[key] = read_script_file(zf, member)
            ids = set().union(*(set(x) for x in maps.values()))
            for audio_id in ids:
                all_records[audio_id] = {
                    "Audio_ID": audio_id,
                    "speaker_id": Path(speaker_file).stem,
                    **{key: maps[key].get(audio_id, "") for key in folders},
                }
    if not all_records:
        raise ValueError("Tidak ada baris data yang berhasil dibaca dari ZIP.")
    return pd.DataFrame(all_records.values()).sort_values("Audio_ID").reset_index(drop=True)

# ---------- NORMALISASI PINYIN ----------
def parse_pinyin_token(token):
    """
    Mengubah token seperti yi1 / lü4 / nv3 menjadi (basis, nada).
    Script LATIC yang diberikan menggunakan notasi angka.
    """
    token = str(token).strip().lower().replace("u:", "v").replace("ü", "v")
    match = re.fullmatch(r"([a-zv]+)([0-5])", token)
    if not match:
        # Notasi bernada tanpa angka atau format tak dikenal tidak ditebak.
        raise ValueError(f"Token Pinyin tidak dikenali: {token!r}")
    base, tone = match.group(1), int(match.group(2))
    # Normalisasi ejaan v/ü dan bentuk j/q/x/y yang umum.
    if re.fullmatch(r"yu(e|an|n)?", base):
        base = "yv" + base[2:]
    elif re.fullmatch(r"[jqx]u(e|an|n)?", base):
        base = base[0] + "v" + base[2:]
    return base, tone

def pinyin_tokens(value):
    return [x for x in str(value).split() if x.strip()]

def hanzi_chars(value):
    # Ambil karakter CJK; tanda baca/spasi tidak dihitung sebagai suku kata.
    return [ch for ch in str(value) if "\u4e00" <= ch <= "\u9fff"]

# ---------- ALIGNMENT SEQUENCE ----------
def align_syllables(reference, observed):

    m, n = len(reference), len(observed)
    cost = np.zeros((m + 1, n + 1), dtype=float)
    back = [[None] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        cost[i, 0], back[i][0] = float(i), "delete"
    for j in range(1, n + 1):
        cost[0, j], back[0][j] = float(j), "insert"

    for i in range(1, m + 1):
        rb, rt = reference[i - 1]
        for j in range(1, n + 1):
            ob, ot = observed[j - 1]
            sub_cost = 0.0 if (rb, rt) == (ob, ot) else (0.25 if rb == ob else 1.25)
            options = [
                (cost[i - 1, j - 1] + sub_cost, "pair"),
                (cost[i - 1, j] + 1.0, "delete"),
                (cost[i, j - 1] + 1.0, "insert"),
            ]
            cost[i, j], back[i][j] = min(options, key=lambda item: item[0])

    pairs, insertions = [None] * m, []
    i, j = m, n
    while i or j:
        op = back[i][j]
        if op == "pair":
            pairs[i - 1] = j - 1
            i -= 1
            j -= 1
        elif op == "delete":
            pairs[i - 1] = None
            i -= 1
        elif op == "insert":
            insertions.append(j - 1)
            j -= 1
        else:
            raise RuntimeError(f"Backtrace alignment gagal pada posisi {(i, j)}")
    return pairs, list(reversed(insertions)), float(cost[m, n])

# ---------- BANGUN DATA TINGKAT AUDIO ----------
source = load_all_scripts(ZIP_PATH)
audio_rows = []
audit_rows = []
parsed = {}

for rec in source.to_dict("records"):
    audio_id = rec["Audio_ID"]
    original_hanzi = hanzi_chars(rec["original_hanzi"])
    testing_hanzi = hanzi_chars(rec["testing_hanzi"])
    ref_raw = pinyin_tokens(rec["original_pinyin"])
    obs_raw = pinyin_tokens(rec["testing_pinyin"])

    try:
        ref = [parse_pinyin_token(x) for x in ref_raw]
        obs = [parse_pinyin_token(x) for x in obs_raw]
    except ValueError as exc:
        audit_rows.append({
            "Audio_ID": audio_id, "status": "PINYIN_PARSE_ERROR",
            "reference_hanzi_count": len(original_hanzi),
            "reference_pinyin_count": len(ref_raw),
            "testing_pinyin_count": len(obs_raw),
            "details": str(exc),
        })
        continue

    if not ref or not obs:
        audit_rows.append({
            "Audio_ID": audio_id, "status": "EMPTY_PINYIN",
            "reference_hanzi_count": len(original_hanzi),
            "reference_pinyin_count": len(ref_raw),
            "testing_pinyin_count": len(obs_raw),
            "details": "Pinyin referensi atau pengujian kosong",
        })
        continue

    pairs, insertions, total_cost = align_syllables(ref, obs)
    omissions = sum(idx is None for idx in pairs)
    matched = [(i, j) for i, j in enumerate(pairs) if j is not None]
    base_mismatch = any(ref[i][0] != obs[j][0] for i, j in matched) or omissions > 0 or bool(insertions)
    tone_mismatch = any(ref[i][1] != obs[j][1] for i, j in matched)

    if base_mismatch:
        scenario = "Incorrect_Phoneme"
        label = 0
    elif tone_mismatch:
        scenario = "Incorrect_Tone"
        label = 0
    else:
        scenario = "Correct_Text"
        label = 1

    hanzi_status = "MATCH" if original_hanzi == testing_hanzi else "DIFFERENT"
    pinyin_status = "MATCH" if ref == obs else "DIFFERENT"

    audio_rows.append({
        "Audio_ID": audio_id,
        "speaker_id": rec["speaker_id"],
        "Original_Text_Hanzi": rec["original_hanzi"],
        "Original_Pinyin_Notasi_Asli": " ".join(ref_raw),
        "Testing_Text_Hanzi": rec["testing_hanzi"],
        "Testing_Pinyin_Notasi_Asli": " ".join(obs_raw),
        "Hanzi_Status": hanzi_status,
        "Pinyin_Status": pinyin_status,
        "Scenario_Type": scenario,
        "Ground_Truth_Label": label,
        "Notes_Error_Type": (
            "Base/segmental mismatch or insertion/omission" if base_mismatch
            else "Tone mismatch" if tone_mismatch
            else "No difference in parsed Pinyin base and tone"
        ),
    })
    parsed[audio_id] = {
        "record": rec, "hanzi": original_hanzi,
        "ref_raw": ref_raw, "obs_raw": obs_raw,
        "ref": ref, "obs": obs,
        "pairs": pairs, "insertions": insertions,
        "edit_cost": total_cost,
    }
    audit_rows.append({
        "Audio_ID": audio_id,
        "status": "ALIGNED",
        "reference_hanzi_count": len(original_hanzi),
        "reference_pinyin_count": len(ref_raw),
        "testing_hanzi_count": len(testing_hanzi),
        "testing_pinyin_count": len(obs_raw),
        "insertions": len(insertions),
        "omissions": omissions,
        "edit_cost": total_cost,
        "details": "",
    })

gt_audio = pd.DataFrame(audio_rows)
if gt_audio.empty:
    raise RuntimeError("Tidak ada audio yang berhasil diproses. Periksa format Pinyin dan file script.")

# ---------- MANIFEST PEMBAGIAN DATA ----------
def assign_split(speaker_id):
    speaker_id = str(speaker_id).zfill(4)
    if speaker_id in {"0011", "0012"}:
        return "finetune_train"
    if speaker_id == "0013":
        return "calibration"
    if speaker_id == "0010":
        return "heldout_test"
    return "unassigned"

manifest = gt_audio.copy()
manifest["split"] = manifest["speaker_id"].map(assign_split)
manifest["speaker_archive"] = manifest["speaker_id"] + ".zip"
manifest["wav_member"] = manifest["Audio_ID"] + ".wav"
manifest = manifest[
    [
        "Audio_ID", "speaker_id", "split", "Scenario_Type", "Ground_Truth_Label",
        "Original_Text_Hanzi", "Original_Pinyin_Notasi_Asli",
        "Testing_Text_Hanzi", "Testing_Pinyin_Notasi_Asli",
        "Hanzi_Status", "Pinyin_Status", "Notes_Error_Type",
        "speaker_archive", "wav_member",
    ]
].copy()

# ---------- GROUND TRUTH TINGKAT SUKU KATA ----------
syllable_rows = []
for audio_id, item in parsed.items():
    ref_raw, obs_raw = item["ref_raw"], item["obs_raw"]
    ref, obs, pairs = item["ref"], item["obs"], item["pairs"]
    hanzi = item["hanzi"]

    # Ground truth per karakter membutuhkan satu Pinyin referensi per Hanzi.
    if len(hanzi) != len(ref_raw):
        audit_rows.append({
            "Audio_ID": audio_id, "status": "HANZI_PINYIN_COUNT_MISMATCH",
            "reference_hanzi_count": len(hanzi),
            "reference_pinyin_count": len(ref_raw),
            "testing_pinyin_count": len(obs_raw),
            "details": "Tidak dibuat label per-suku-kata karena jumlah Hanzi dan Pinyin referensi berbeda.",
        })
        continue

    for idx, (char, target, obs_idx) in enumerate(zip(hanzi, ref_raw, pairs)):
        if obs_idx is None:
            syllable_rows.append({
                "id": audio_id, "syllable_index": idx, "hanzi": char,
                "target_pinyin": target, "observed_pinyin": None,
                "gop_label": 0, "got_label": np.nan,
                "alignment_status": "TARGET_SYLLABLE_OMITTED_IN_TESTING",
                "label_source": "Original/Testing Pinyin; automated sequence alignment",
                "script_expert_review": "not_verified",
                "per_hanzi_audio_rating": False,
            })
            continue

        observed = obs_raw[obs_idx]
        ref_base, ref_tone = ref[idx]
        obs_base, obs_tone = obs[obs_idx]
        gop_label = int(ref_base == obs_base)
        got_label = int(ref_tone == obs_tone)

        if gop_label and got_label:
            status = "MATCH_BASE_AND_TONE"
        elif gop_label:
            status = "TONE_MISMATCH_ONLY"
        elif got_label:
            status = "SEGMENTAL_MISMATCH_ONLY"
        else:
            status = "SEGMENTAL_AND_TONE_MISMATCH"

        syllable_rows.append({
            "id": audio_id, "syllable_index": idx, "hanzi": char,
            "target_pinyin": target, "observed_pinyin": observed,
            "gop_label": gop_label, "got_label": got_label,
            "alignment_status": status,
            "label_source": "Original/Testing Pinyin; automated sequence alignment",
            "script_expert_review": "not_verified",
            "per_hanzi_audio_rating": False,
        })

gt_syllable = pd.DataFrame(syllable_rows)
audit = pd.DataFrame(audit_rows)

# ---------- SIMPAN OUTPUT ----------
paths = {
    "audio": OUTPUT_DIR / "ground_truth_perbandingan_per_ID.csv",
    "manifest": OUTPUT_DIR / "latic_finetuning_split_manifest.csv",
    "syllable": OUTPUT_DIR / "latic_syllable_ground_truth_script_aligned.csv",
    "audit": OUTPUT_DIR / "latic_script_alignment_audit.csv",
}
gt_audio.to_csv(paths["audio"], index=False, encoding="utf-8-sig")
manifest.to_csv(paths["manifest"], index=False, encoding="utf-8-sig")
gt_syllable.to_csv(paths["syllable"], index=False, encoding="utf-8-sig")
audit.to_csv(paths["audit"], index=False, encoding="utf-8-sig")

print("Selesai membentuk file:")
for key, path in paths.items():
    print(f"- {path}: {sum(1 for _ in open(path, encoding='utf-8-sig')) - 1:,} baris")
print("\nDistribusi Scenario_Type:")
print(gt_audio["Scenario_Type"].value_counts(dropna=False).to_string())
print("\nDistribusi split:")
print(manifest["split"].value_counts(dropna=False).to_string())
print("\nDistribusi status alignment:")
print(audit["status"].value_counts(dropna=False).to_string())
print("\nCatatan: label ini berbasis perbandingan script Pinyin.")


Selesai membentuk file:
- latic_ground_truth_output/ground_truth_perbandingan_per_ID.csv: 2,579 baris
- latic_ground_truth_output/latic_finetuning_split_manifest.csv: 2,579 baris
- latic_ground_truth_output/latic_syllable_ground_truth_script_aligned.csv: 33,803 baris
- latic_ground_truth_output/latic_script_alignment_audit.csv: 2,587 baris

Distribusi Scenario_Type:
Scenario_Type
Correct_Text         1613
Incorrect_Phoneme     918
Incorrect_Tone         48

Distribusi split:
split
finetune_train    1295
calibration        647
heldout_test       637

Distribusi status alignment:
status
ALIGNED                        2579
HANZI_PINYIN_COUNT_MISMATCH       8

Catatan: label ini berbasis perbandingan script Pinyin.
